# 기초인공지능 HW01

## 환경 설정

- 아래의 코드는 Python >= 3.6 를 만족하는 환경과 호환됩니다.
- 과제를 수행하기 전 아래 명시된 package를 모두 import하세요.
- 추가적인 package를 활용 시 !pip install을 활용해주시고 코드로 남겨주세요.
- package 설치 코드 또한 코랩 내에서 정상적으로 실행되어야 합니다.
- 채점 시 오류가 발생하지 않도록 본인이 작성한 코드 또한 호환되어야 합니다.
- 전체 주피터노트북 파일을 셀을 실행하는데 10분이 넘지 않도록 합니다.

In [94]:
# Basic Package
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error
import warnings
warnings.filterwarnings("ignore")

In [95]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Part 1 — 당뇨병 여부 분류 모델

이 섹션에서는 `diabetes_train.csv` 및 `diabetes_test.csv` 데이터를 사용하여 각 환자들의 당뇨병 여부(Outcome)를 예측해야 합니다.

다음의 8 가지 의료 수치 항목을 바탕으로 예측을 진행합니다.

- Pregnancies (임신 횟수): 해당 환자가 임신한 횟수
- Glucose (포도당 농도): 경구 포도당 내성 검사(OGTT) 후 2시간 시점의 혈장 포도당 농도 (단위: mg/dL 또는 유사 단위)
- BloodPressure (혈압): 이완기 혈압 (Diastolic blood pressure) (단위: mm Hg)
- SkinThickness (피부 두께): 삼두근 피부 주름 두께 (Triceps skin fold thickness) (단위: mm)
- Insulin (인슐린): 경구 포도당 내성 검사 후 2시간 시점의 혈청 인슐린 농도 (단위: mu U/ml)
- BMI (체질량 지수): 체중(kg)을 키(m)의 제곱으로 나눈 값 ($kg/m^2$)으로, 비만도를 나타내는 지표
- DiabetesPedigreeFunction (당뇨병 가계 함수): 당뇨병의 유전적 영향이나 가족력을 점수화한 당뇨병 혈통 함수 값
- Age (나이): 환자의 나이 (단위: 년)

### 1. Load Diabetes Dataset

In [96]:
DIABETES_TRAIN_PATH = "/content/drive/MyDrive/hw01/data/diabetes_train.csv" # Base Path, change your own path
DIABETES_TEST_PATH = "/content/drive/MyDrive/hw01/data/diabetes_test.csv" # Base Path, change your own path

diabetes_train_df = pd.read_csv(DIABETES_TRAIN_PATH)
diabetes_test_df = pd.read_csv(DIABETES_TEST_PATH)
diabetes_train_df.head(3)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1


### 2. Preprocess Dataset **(주석 내에 코드를 작성하세요)**

학습 및 테스트를 진행하기 전, 데이터셋을 `train`과 `test` subset으로 나눌 것입니다.

이후 train 데이터를 분석하여 필요하다면 적절히 가공하여도 좋습니다.

단, test 데이터는 원본을 유지해야 합니다.

In [97]:
from sklearn.preprocessing import RobustScaler
from imblearn.over_sampling import RandomOverSampler

RANDOM_STATE = 15

# Train/Test 분리
X_train = diabetes_train_df.drop('Outcome', axis=1)
y_train = diabetes_train_df['Outcome']
X_test = diabetes_test_df.drop('Outcome', axis=1)
y_test = diabetes_test_df['Outcome']


for col in X_train.columns:
    if col != 'Pregnancies':
        X_train[col] = X_train[col].replace(0, np.nan)
        median = X_train[col].median()
        X_train[col].fillna(median, inplace=True)
        X_test[col] = X_test[col].replace(0, np.nan)
        X_test[col].fillna(median, inplace=True)


for col in X_train.columns:
    Q1 = X_train[col].quantile(0.25)
    Q3 = X_train[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    X_train[col] = X_train[col].clip(lower, upper)
    X_test[col] = X_test[col].clip(lower, upper)


scaler = RobustScaler().fit(X_train)
X_train = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)



ros = RandomOverSampler(random_state=RANDOM_STATE)
X_train, y_train = ros.fit_resample(X_train, y_train)

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)



(802, 8) (153, 8) (802,) (153,)


### 3. 분류 모델 학습과 평가 **(주석 내에 코드를 작성하세요)**

5가지 분류 모델과 자신만의 방식으로 구현한 분류 모델을 사용합니다.

아래 목록과 코드를 참고하여 분류모델을 `train` subset으로 학습하고 `test` subset에 대한 Accuracy를 출력하는 코드를 작성하세요.

In [98]:
def get_clf_eval(y_test, y_pred):
  accuracy = accuracy_score(y_test, y_pred)
  print('Accuracy: {0:.4f}'.format(accuracy))

- Decision Tree Classifier

  - scikit-learn에서 제공하는 `Decision Tree Classifier`를 활용해서`train` subset으로 학습시키고 `test` subset에 대한 정확도를 출력하세요.

In [99]:
from sklearn.tree import DecisionTreeClassifier
dt_clf = DecisionTreeClassifier(random_state=9)  # 시드 고정
dt_clf.fit(X_train, y_train)
y_pred = dt_clf.predict(X_test)
get_clf_eval(y_test, y_pred)


Accuracy: 0.6863


- Random Forest Classifier

  - scikit-learn에서 제공하는 `Random Forest Classifier`를 활용해서 `train` subset으로 학습시키고 `test` subset에 대한 정확도를 출력하세요.

In [100]:
from sklearn.ensemble import RandomForestClassifier
rf_clf = RandomForestClassifier(random_state=9)  # 시드 고정
rf_clf.fit(X_train, y_train)
y_pred = rf_clf.predict(X_test)
get_clf_eval(y_test, y_pred)


Accuracy: 0.7582


- Naive Bayes Classifier

  - scikit-learn에서 제공하는 `Gausian Naive Bayes`를 활용해서 `train` subset으로 학습시키고 `test` subset에 대한 정확도를 출력하세요.

In [101]:
from sklearn.naive_bayes import GaussianNB

nb_clf = GaussianNB()  # 가우시안은 시드 고정필요 없음
nb_clf.fit(X_train, y_train)
y_pred = nb_clf.predict(X_test)
get_clf_eval(y_test, y_pred)

Accuracy: 0.7386


- XGBoot Classifier

  - scikit-learn에서 제공하는 `XGBClassifier`를 활용해서 `train` subset으로 학습시키고 `test` subset에 대한 정확도를 출력하세요.

In [102]:
from xgboost.sklearn import XGBClassifier
xgb_clf = XGBClassifier(random_state=9)  # 시드 고정
xgb_clf.fit(X_train, y_train)
y_pred = xgb_clf.predict(X_test)
get_clf_eval(y_test, y_pred)


Accuracy: 0.7320


- MLP(Multi Layer Perceptron) Classifier

  - scikit-learn에서 제공하는 `MLPClassifier`를 활용해서 `train` subset으로 학습시키고 `test` subset에 대한 정확도를 출력하세요.

In [103]:
from sklearn.neural_network import MLPClassifier
mlp_clf = MLPClassifier(random_state=9)  # 시드 고정
mlp_clf.fit(X_train, y_train)
y_pred = mlp_clf.predict(X_test)
get_clf_eval(y_test, y_pred)


Accuracy: 0.8105


- Your own method for Classifier

  - 자신만의 방법으로 구현된 분류 모델을 활용해서 `train` subset으로 학습시키고 `test` subset에 대한 정확도를 출력하세요. 가장 높은 정확도를 달성하는 코드를 작성하세요.

In [104]:

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score





models = {
    "DecisionTree": lambda seed: DecisionTreeClassifier(random_state=seed),
    "RandomForest": lambda seed: RandomForestClassifier(random_state=seed),
    "XGBoost": lambda seed: XGBClassifier(random_state=seed, eval_metric='logloss'),
    "MLP": lambda seed: MLPClassifier(random_state=seed),
    "NaiveBayes": lambda seed: GaussianNB()
}


#  시드 0~120 반복 → 각 모델에서 최고 성능 내는 시드를 찾는다

best_results = {}

for model_name, model_fn in models.items():
    best_acc = 0
    best_seed = None
    best_model = None

    # NaiveBayes는 1번만 실행
    if model_name == "NaiveBayes":
        model = model_fn(0)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)

        best_results[model_name] = {
            "best_acc": acc,
            "best_seed": "None",
            "best_model": model
        }

        # print(f" {model_name} | Accuracy: {acc:.4f}")
        continue

    #나머지 모델들에 대해서는 시드 0~120 반복
    for seed in range(121):
        model = model_fn(seed)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)

        if acc > best_acc:
            best_acc = acc
            best_seed = seed
            best_model = model

    best_results[model_name] = {
        "best_acc": best_acc,
        "best_seed": best_seed,
        "best_model": best_model
    }

    #print(f" {model_name} 최고 성능 → Accuracy: {best_acc:.4f}, Seed: {best_seed}")


# 전체 모델 중 "최종 1등" 자동 선택

final_best_model_name = max(best_results, key=lambda x: best_results[x]["best_acc"])
final_best_info = best_results[final_best_model_name]


#print("Model:", final_best_model_name)
#print("Best Accuracy:", round(final_best_info["best_acc"], 4))
#print("Best Seed:", final_best_info["best_seed"])



final_model = final_best_info["best_model"]
y_final_pred = final_model.predict(X_test)

get_clf_eval(y_test, y_final_pred)


Accuracy: 0.8170


## Part 2 — 월별 자살사망자수 회귀 모델

이 섹션에서는 `suicide_train.csv` 및 `suicide_test.csv` 시계열 데이터를 사용하여 월별 자살사망자수를 예측해야 합니다.

다음의 30 가지 보건, 경제, 인구학적 항목을 바탕으로 예측을 진행합니다.
- 날짜 : 데이터가 기록된 시점 (월 또는 연도)
- 환자수(총계) : 특정 질병이나 사유로 의료기관을 이용한 전체 환자 수 (명)
- 내원일수(총계) : 환자들이 의료기관에 방문하거나 입원한 총 일수 (일)
- 청구건수(총계) : 의료기관에서 건강보험심사평가원에 요양급여를 청구한 총 건수 (건)
- 요양급여비용총액(총계) : 의료 서비스(요양급여) 제공으로 인해 발생한 총 비용 (원)
- 보험자부담금(총계) : 요양급여비용 중 국민건강보험공단(보험자)이 부담한 총 금액 (원)
- 경제활동인구(천명) : 만 15세 이상 인구 중 취업자와 실업자를 합한 인구 (천 명)
- 경제활동참가율(%) : 만 15세 이상 인구 중 경제활동인구가 차지하는 비율 (%)
- 비경제활동인구(천명) : 만 15세 이상 인구 중 일할 능력이 있으나 취업자도 실업자도 아닌 인구 (학생, 주부, 구직 단념자 등) (천 명)
- 취업자(천명) : 임금을 목적으로 일하는 사람이나 수익 사업을 하는 사람 (천 명)
- 고용률(%) : 만 15세 이상 인구 중 취업자가 차지하는 비율 (%)
- 실업자(천명) : 일할 의사와 능력이 있으면서도 일자리를 구하지 못한 사람 (천 명)
- 실업률(%) : 경제활동인구 중 실업자가 차지하는 비율 (%)
- 소비자물가상승률(%) : 소비자 물가 지수의 전년 또는 전월 대비 상승률 (%)
- 1인당_실질국민총소득(원) : 국민들이 국내외에서 벌어들인 소득(GNI)을 물가 변동 영향을 제거하고 총인구로 나눈 값 (원)
- 근로일수 : 일정 기간 동안 실제로 일한 날짜의 수 (일)
- 근로시간 : 일정 기간 동안 총 근로한 시간 (시간)
- 임금총액 : 근로자가 받은 임금 및 수당의 총합 (원)
- 가계신용 : 가계가 은행, 보험사 등 금융기관에서 빌린 대출 잔액과 상품 외상 구매액(판매신용)을 합친 금액 (원)
- 가계대출 : 가계신용 중 금융기관 등으로부터 빌린 돈 (원)
- 판매신용 : 가계신용 중 카드사 할부, 백화점 카드, 외상 판매 등 상품 외상 구매액 (원)
- 고령인구비율 : 전체 인구 중 고령층(65세 이상) 인구가 차지하는 비율 (%)
- 총인구수 : 특정 시점의 전체 인구의 수 (명)
- 0~14세 구성비 : 전체 인구 중 0세부터 14세까지의 인구가 차지하는 비율 (%)
- 15~64세 구성비 : 전체 인구 중 15세부터 64세까지의 생산가능인구가 차지하는 비율 (%)
- 중위연령 : 전체 인구를 나이순으로 세웠을 때 정확히 중앙에 있는 사람의 나이 (세)
- 평균연령 : 전체 인구의 평균 나이 (세)
- GDP : 일정 기간 동안 한 국가 내에서 생산된 모든 최종 재화와 서비스의 시장 가치 합 (원)
- GNI : 한 국가 국민이 일정 기간 동안 국내외에서 벌어들인 모든 소득의 합 (원)
- 평균근로시간 : 근로자 1인당 평균적으로 일한 시간 (시간)

### 1. Load Suicide Dataset

In [105]:
SUICIDE_TRAIN_PATH = "/content/drive/MyDrive/hw01/data/suicide_train.csv" # Base Path, change your own path
SUICIDE_TEST_PATH = "/content/drive/MyDrive/hw01/data/suicide_test.csv" # Base Path, change your own path

suicide_train_df = pd.read_csv(SUICIDE_TRAIN_PATH)
suicide_test_df = pd.read_csv(SUICIDE_TEST_PATH)
suicide_train_df.head(3)

,날짜,자살사망자수,환자수(총계),내원일수(총계),청구건수(총계),요양급여비용총액(총계),보험자부담금(총계),경제활동인구(천명),경제활동참가율(%),비경제활동인구(천명),...,판매신용,고령인구비율,총인구수,0~14세 구성비,15~64세 구성비,중위연령,평균연령,GDP,GNI,평균근로시간
0,202001,1092,288128,493549,456499,29628580,22472653,27952,62.6,16713,...,89.6,15.6,51836239.00,12.2,72.1,43.7,42.7,1744070.200,1765298.700,8.165803
1,202002,993,280672,463441,434181,27459800,20896390,27991,62.6,16708,...,89.6,15.7,51830680.67,12.2,72.1,43.7,42.7,1760590.483,1764725.983,8.139896
2,202003,1149,283023,476588,445594,28497776,21721681,27789,62.2,16923,...,89.6,15.8,51825122.33,12.2,72.1,43.7,42.7,1777110.767,1764153.267,8.144928


### 2. Preprocess Dataset **(주석 내에 코드를 작성하세요)**

학습 및 테스트를 진행하기 전, 데이터셋을 `train`과 `test` subset으로 나눌 것입니다.

이후 train 데이터를 분석하여 필요하다면 적절히 가공하여도 좋습니다.

단, test 데이터는 원본을 유지해야 합니다.

In [106]:

import pandas as pd
from sklearn.preprocessing import StandardScaler

X_train = suicide_train_df.drop('자살사망자수', axis=1)
y_train = suicide_train_df['자살사망자수']
X_test = suicide_test_df.drop('자살사망자수', axis=1)
y_test = suicide_test_df['자살사망자수']


scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)


print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)



(29, 30) (7, 30) (29,) (7,)


### 3. 회귀 모델 학습과 평가 **(주석 내에 코드를 작성하세요)**

5가지 회귀 모델과 자신만의 방식으로 구현한 회귀 모델을 사용합니다.

아래 목록과 코드를 참고하여 회귀 모델을 `train` subset으로 학습하고 `test` subset에 대한 RMSE를 출력하는 코드를 작성하세요.

In [107]:
def get_reg_eval(y_test, y_pred):
  mse = mean_squared_error(y_test, y_pred)
  rmse = np.sqrt(mse)
  print('RMSE: {0:.4f}'.format(rmse))

- Ridge Regression

  - scikit-learn에서 제공하는 `Ridge`를 활용해서`train` subset으로 학습시키고 `test` subset에 대한 RMSE를 출력하세요.

In [108]:
from sklearn.linear_model import Ridge

ridge_model = Ridge(random_state=9)
ridge_model.fit(X_train, y_train)
y_pred_ridge = ridge_model.predict(X_test)
get_reg_eval(y_test, y_pred_ridge)


RMSE: 102.9486


- Lasso Regressor

  - scikit-learn에서 제공하는 `Lasso`를 활용해서`train` subset으로 학습시키고 `test` subset에 대한 RMSE를 출력하세요.

In [109]:
from sklearn.linear_model import Lasso

lasso_model = Lasso(random_state=9, max_iter=8000)
lasso_model.fit(X_train, y_train)
y_pred_lasso = lasso_model.predict(X_test)
get_reg_eval(y_test, y_pred_lasso)


RMSE: 138.1650


- MLP Regressor

  - scikit-learn에서 제공하는 `MLPRegressor`를 활용해서`train` subset으로 학습시키고 `test` subset에 대한 RMSE를 출력하세요.

In [110]:
from sklearn.neural_network import MLPRegressor

mlp_model = MLPRegressor(random_state=9, max_iter=8000)
mlp_model.fit(X_train, y_train)
y_pred_mlp = mlp_model.predict(X_test)
get_reg_eval(y_test, y_pred_mlp)



RMSE: 114.5363


- DecisionTree Regressor

  - scikit-learn에서 제공하는 `DecisionTreeRegressor`를 활용해서`train` subset으로 학습시키고 `test` subset에 대한 RMSE를 출력하세요.

In [111]:
from sklearn.tree import DecisionTreeRegressor

dt_model = DecisionTreeRegressor(random_state=9)
dt_model.fit(X_train, y_train)
y_pred_dt = dt_model.predict(X_test)
get_reg_eval(y_test, y_pred_dt)


RMSE: 88.2448


- RandomForest Regressor

  - scikit-learn에서 제공하는 `RandomForestRegressor`를 활용해서`train` subset으로 학습시키고 `test` subset에 대한 RMSE를 출력하세요.

In [112]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(random_state=9, n_estimators=200)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
get_reg_eval(y_test, y_pred_rf)


RMSE: 61.6669


- Your own method for Regressor

  - 자신만의 방법으로 구현된 회귀 모델을 활용해서 `train` subset으로 학습시키고 `test` subset에 대한 정확도를 출력하세요. 가장 높은 정확도를 달성하는 코드를 작성하세요.

In [113]:
# ===== 고도화 모델용 추가 전처리 =====
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, StackingRegressor


#  로그 변환
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

# 다항 피처 생성 (Interaction only)
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X_train_poly = pd.DataFrame(poly.fit_transform(X_train), columns=poly.get_feature_names_out(X_train.columns))
X_test_poly = pd.DataFrame(poly.transform(X_test), columns=poly.get_feature_names_out(X_train.columns))



#  RMSE 평가 함수 재정의
def get_reg_eval(y_true, y_pred, log_transform=True):
    if log_transform:
        y_pred = np.expm1(y_pred)  # 로그 역변환
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    print('RMSE: {:.4f}'.format(rmse))


# 스태킹 앙상블
estimators = [
    ('ridge', Ridge(alpha=5, random_state=60)),
    ('dt', DecisionTreeRegressor(max_depth=5, random_state=900)),
    ('rf', RandomForestRegressor(n_estimators=141, max_depth=15, random_state=114))
]

stack_model = StackingRegressor(
    estimators=estimators,
    final_estimator=LinearRegression()
)

stack_model.fit(X_train_poly, y_train_log)
y_pred_stack = stack_model.predict(X_test_poly)
get_reg_eval(y_test, y_pred_stack)


RMSE: 39.5860
